# A6 — SAH bonus

Surface Area Heuristic instead of the naive midpoint split.

Buckets on the longest axis, cost = C_trav + (SA_L/SA * N_L + SA_R/SA * N_R) * C_isect.

## Timings on the bunny (10k tri)

| Build | Time / frame |
|---|---|
| naive iterate | 34.2 s |
| BVH midpoint | 0.81 s |
| BVH + SAH    | 0.63 s |


In [1]:
import numpy as np

def surface_area(pmin, pmax):
    d = pmax - pmin
    return 2 * (d[0]*d[1] + d[1]*d[2] + d[0]*d[2])

def sah_split(prims, pmin, pmax):
    N = len(prims)
    if N <= 4:
        return None  # leaf
    axis = int(np.argmax(pmax - pmin))
    B = 12  # buckets
    centroids = np.array([p.centroid[axis] for p in prims])
    lo, hi = centroids.min(), centroids.max()
    if hi - lo < 1e-9:
        return None
    buckets = [[[], np.full(3, np.inf), np.full(3, -np.inf)] for _ in range(B)]
    for p, c in zip(prims, centroids):
        b = min(B - 1, int(B * (c - lo) / (hi - lo)))
        buckets[b][0].append(p)
        buckets[b][1] = np.minimum(buckets[b][1], p.pmin)
        buckets[b][2] = np.maximum(buckets[b][2], p.pmax)
    sa = surface_area(pmin, pmax)
    best_cost, best_i = np.inf, None
    for i in range(1, B):
        L = buckets[:i]; R = buckets[i:]
        nL = sum(len(b[0]) for b in L)
        nR = sum(len(b[0]) for b in R)
        if nL == 0 or nR == 0:
            continue
        lmin = np.min([b[1] for b in L if len(b[0])], axis=0)
        lmax = np.max([b[2] for b in L if len(b[0])], axis=0)
        rmin = np.min([b[1] for b in R if len(b[0])], axis=0)
        rmax = np.max([b[2] for b in R if len(b[0])], axis=0)
        cost = 0.125 + (surface_area(lmin, lmax) * nL + surface_area(rmin, rmax) * nR) / sa
        if cost < best_cost:
            best_cost, best_i = cost, i
    return axis, best_i
